In [ ]:
# Parameters
WAREHOUSE_CONN = ""  # ODBC: Driver={ODBC Driver 18 for SQL Server};Server=<host>.datawarehouse.fabric.microsoft.com,...
SCALE_SIZE = "medium"  # small | medium | large | fabric_demo
AUTH_METHOD = "cli"    # cli | msi | spn
SEED = 42


In [ ]:
from sqllocks_spindle import Spindle
from sqllocks_spindle.domains.financial import FinancialDomain
from sqllocks_spindle.fabric.sql_database_writer import FabricSqlDatabaseWriter

if not WAREHOUSE_CONN:
    raise ValueError("Set WAREHOUSE_CONN to your Fabric Warehouse ODBC connection string")

spindle = Spindle()
result = spindle.generate(domain=FinancialDomain(), scale=SCALE_SIZE, seed=SEED)
print(f"Generated {sum(len(df) for df in result.tables.values()):,} rows")
for t, df in result.tables.items():
    print(f"  {t}: {len(df):,} rows")


In [ ]:
errors = result.verify_integrity()
assert errors == [], f"FK integrity errors: {errors}"
print("FK integrity: PASS")


In [ ]:
writer = FabricSqlDatabaseWriter(WAREHOUSE_CONN, auth_method=AUTH_METHOD)
writer.write(result, schema_name="dbo", mode="create_insert")
print("Write to Warehouse: DONE")


In [ ]:
# Validate — query Warehouse row counts
import pyodbc
import struct
from azure.identity import AzureCliCredential

cred = AzureCliCredential()
token = cred.get_token("https://database.windows.net/.default").token
token_bytes = token.encode("utf-16-le")
token_struct = struct.pack(f"<I{len(token_bytes)}s", len(token_bytes), token_bytes)

conn = pyodbc.connect(WAREHOUSE_CONN, attrs_before={1256: token_struct})
cursor = conn.cursor()
for table_name in result.tables:
    cursor.execute(f"SELECT COUNT(*) FROM dbo.[{table_name}]")
    count = cursor.fetchone()[0]
    expected = len(result.tables[table_name])
    status = "OK" if count == expected else "MISMATCH"
    print(f"  {status} {table_name}: {count:,} rows")
conn.close()
